# ROOT TH2 maps in Square-Dalitz coordinates

For B decays, efficiency/background maps are often stored directly in $(m',\theta')$. The ROOT helpers convert invariant coordinates internally, while `generate_toy` and `FitSession` use the maps exactly like ordinary efficiencies/backgrounds.


In [ ]:
import numpy as np
import uproot
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    BackgroundSpec,DecayChannel,DecayModel,FitSession,NonResonant,Parameter,RealImag,
    ToyBackground,enable_x64,generate_toy,plot_dalitz,plot_square_dalitz,
    square_dalitz_background_from_root,square_dalitz_efficiency_from_root,
)
enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [NonResonant(RealImag(1.0,0.0))],
    normalization_method="square-dalitz",normalization_resolution=160,normalization_pair=(0,2),
)
edges=np.linspace(0,1,26)
c=0.5*(edges[:-1]+edges[1:])
MP,TP=np.meshgrid(c,c,indexing="ij")
eff_values=0.45+0.35*MP+0.10*np.cos(np.pi*TP)
bkg_values=0.30+0.70*TP

with uproot.recreate("maps_sdp.root") as f:
    f["efficiency_sdp"]=(eff_values,edges,edges)
    f["background_sdp"]=(bkg_values,edges,edges)

kwargs=dict(
    mother_mass=model.channel.parent_mass,
    masses=model.channel.daughter_masses,
    pair=(0,2),
)
eff=square_dalitz_efficiency_from_root("maps_sdp.root","efficiency_sdp",**kwargs)
bkg=square_dalitz_background_from_root("maps_sdp.root","background_sdp",**kwargs)


In [ ]:
f_sig=Parameter("signal_fraction",0.76,bounds=(0.05,0.99))
data=generate_toy(
    model,25_000,efficiency=eff,signal_fraction=0.82,
    backgrounds=(ToyBackground("comb",bkg),),
    seed=1515,pool_size=180_000,
)
plot_square_dalitz(data,**kwargs,title="Toy in Square Dalitz")
plt.show()
plot_dalitz(data,x="s13",y="s23",title="Same toy in ordinary Dalitz")
plt.show()

session=FitSession(
    model,data,efficiency=eff,signal_fraction=f_sig,
    backgrounds=(BackgroundSpec("comb",bkg),),
)
result=session.fit()
session.report(result)
session.plot_projection(result,"s13")
plt.show()
